# EcoNexus — Offline Evaluation of Confidence-Gated Routing

This notebook tests the confidence-gated routing policy from the EcoNexus paper **on images**, without the physical sorting line.

**What it does**
1. Downloads TrashNet and merges its six classes into the paper's three: **paper** (paper + cardboard), **plastic**, **general** (glass + metal + trash).
2. Makes stratified train / validation / test splits (70 / 15 / 15). The **test split is never used for training, temperature fitting or threshold choice.**
3. Fine-tunes **YOLOv8n-cls** (the same architecture as the robot).
4. Evaluates the classifier: confusion matrix, per-class precision / recall / F1.
5. Runs the routing policy **ungated** and **gated** at several thresholds τ, with raw softmax confidence and after **temperature scaling**.
6. Chooses τ on the validation split by minimising expected cost for a stated cost ratio, then reports it once on the test split.
7. Repeats steps 2–6 for several random seeds and reports mean ± standard deviation.
8. Saves every table (CSV + Markdown) and figure (PNG, 300 dpi) and zips them for download.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. With 3 seeds × 30 epochs expect roughly 30–60 minutes.

**How to describe this in the paper:** this is an *offline evaluation on a public dataset*. It is not a measurement on the physical line, whose camera, lighting and item presentation differ from TrashNet. State that clearly.

In [ ]:
!pip -q install ultralytics datasets tabulate
import os, json, random, shutil, time, zipfile, glob
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.optimize import minimize_scalar
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, accuracy_score
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())

## 1. Configuration
Change these values here only; every later cell reads them.

In [ ]:
SEEDS        = [0, 1, 2]          # independent split + training runs (repeated trials)
EPOCHS       = 30
IMGSZ        = 224
BATCH        = 64
CLASSES      = ['general', 'paper', 'plastic']   # alphabetical = YOLO folder order
RECYCLABLE   = ['paper', 'plastic']              # streams the gate protects
MERGE = {                                        # TrashNet class -> EcoNexus class
    'cardboard': 'paper', 'paper': 'paper',
    'plastic': 'plastic',
    'glass': 'general', 'metal': 'general', 'trash': 'general',
}
THRESHOLDS   = [0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]   # 0.0 = ungated
COST_RATIOS  = [1, 3, 9]          # k = L_c / L_r ; theoretical tau* = k / (k + 1)
K_MAIN       = 9                  # cost ratio used for the headline result (state it in the paper)
OUT = Path('outputs'); OUT.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300, 'font.size': 10})

## 2. Download TrashNet and merge classes

In [ ]:
RAW = Path('trashnet_raw')
def download_trashnet():
    if RAW.exists() and any(RAW.iterdir()):
        return
    try:
        from datasets import load_dataset
        ds = load_dataset('garythung/trashnet', split='train')
        names = ds.features['label'].names
        print('Hugging Face TrashNet classes:', names)
        for i, ex in enumerate(ds):
            c = names[ex['label']]
            d = RAW / c; d.mkdir(parents=True, exist_ok=True)
            ex['image'].convert('RGB').save(d / f'{c}_{i:05d}.jpg', quality=95)
    except Exception as e:
        print('Hugging Face download failed (', e, ') - trying the original GitHub release')
        !wget -q -O trashnet.zip https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip
        !unzip -q -o trashnet.zip -d trashnet_zip
        src = Path('trashnet_zip/dataset-resized')
        for c in MERGE:
            shutil.copytree(src / c, RAW / c, dirs_exist_ok=True)
download_trashnet()

items = []   # (path, original class, merged class)
for c in sorted(os.listdir(RAW)):
    assert c in MERGE, f'Unexpected TrashNet class {c!r} - add it to MERGE'
    for p in sorted((RAW / c).glob('*.jpg')):
        items.append((str(p), c, MERGE[c]))
df_all = pd.DataFrame(items, columns=['path', 'orig', 'cls'])
print(len(df_all), 'images')
display(pd.crosstab(df_all.orig, df_all.cls, margins=True))

## 3. Stratified splits (one per seed)
Stratified on the *original* TrashNet class, so each split keeps the same mix of cardboard vs paper, glass vs metal vs trash.

In [ ]:
def make_split(seed):
    root = Path(f'data_seed{seed}')
    tr, rest = train_test_split(df_all, test_size=0.30, stratify=df_all.orig, random_state=seed)
    va, te = train_test_split(rest, test_size=0.50, stratify=rest.orig, random_state=seed)
    if root.exists(): shutil.rmtree(root)
    for name, part in [('train', tr), ('val', va), ('test', te)]:
        for _, r in part.iterrows():
            d = root / name / r.cls; d.mkdir(parents=True, exist_ok=True)
            os.symlink(os.path.abspath(r.path), d / Path(r.path).name)
    return root, {'train': tr, 'val': va, 'test': te}

_, parts0 = make_split(SEEDS[0])
counts = pd.DataFrame({k: v.cls.value_counts() for k, v in parts0.items()}).loc[CLASSES]
counts.loc['total'] = counts.sum()
counts.to_csv(OUT / 'table_dataset_split.csv'); display(counts)

## 4. Fine-tune YOLOv8n-cls and collect predictions

In [ ]:
from ultralytics import YOLO

def train_seed(seed, root):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    m = YOLO('yolov8n-cls.pt')
    m.train(data=str(root), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=seed, deterministic=True,
            project='runs', name=f'seed{seed}', exist_ok=True, verbose=False, plots=False)
    return YOLO(f'runs/seed{seed}/weights/best.pt')

def predict_logp(model, part):
    """Returns log-probabilities (N x 3, columns in CLASSES order) and integer labels."""
    names = model.names                       # {idx: name}
    order = [ [k for k, v in names.items() if v == c][0] for c in CLASSES ]
    P, t0 = [], time.time()
    for r in model.predict(list(part.path), imgsz=IMGSZ, stream=True, verbose=False):
        P.append(r.probs.data.cpu().numpy()[order])
    P = np.clip(np.array(P, dtype=np.float64), 1e-12, 1.0)
    y = np.array([CLASSES.index(c) for c in part.cls])
    return np.log(P), y, (time.time() - t0) / len(part)

## 5. Analysis functions

**Routing rule** (paper, Section III-D). Let ĉ be the predicted class and p its confidence. If ĉ ∈ {paper, plastic} and p < τ, route to general; otherwise route to ĉ. With τ = 0 the gate is off.

**Definitions used in every table**
- *Admitted*: items routed into a recycling stream (paper or plastic).
- *Contaminant*: an admitted item whose true class is not that stream (including paper routed to plastic and vice versa).
- *Contamination %*: contaminants / admitted. *Purity* = 100 − contamination.
- *Intercepted*: contaminants removed compared with the ungated run on the same items.
- *Recyclables lost*: items whose true class is paper or plastic that end in general. *Lost to gate* counts only those the classifier got right but the gate diverted.
- *Expected cost* with ratio k = L_c / L_r: k × contaminants + recyclables lost (in units of L_r).
- Temperature T is fitted on the validation split only by minimising negative log-likelihood. Calibrated confidence = softmax(log p / T).

In [ ]:
def softmax_T(logp, T=1.0):
    z = logp / T
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def nll(logp, y, T=1.0):
    P = softmax_T(logp, T)
    return float(-np.mean(np.log(np.clip(P[np.arange(len(y)), y], 1e-12, 1))))

def fit_temperature(logp, y):
    r = minimize_scalar(lambda lt: nll(logp, y, np.exp(lt)), bounds=(-3, 3), method='bounded')
    return float(np.exp(r.x))

def ece(P, y, bins=15):
    conf, pred = P.max(1), P.argmax(1)
    edges = np.linspace(0, 1, bins + 1); e = 0.0; rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            acc, c = (pred[m] == y[m]).mean(), conf[m].mean()
            e += m.mean() * abs(acc - c); rows.append((c, acc, m.sum()))
    return float(e), rows

RIDX = [CLASSES.index(c) for c in RECYCLABLE]
GEN = CLASSES.index('general')

def route(P, tau):
    pred, conf = P.argmax(1), P.max(1)
    routed = pred.copy()
    routed[np.isin(pred, RIDX) & (conf < tau)] = GEN
    return routed, pred, conf

def gate_metrics(P, y, tau, k=None):
    routed, pred, _ = route(P, tau)
    adm = np.isin(routed, RIDX)
    cont = adm & (routed != y)
    rec_true = np.isin(y, RIDX)
    lost = rec_true & (routed == GEN)
    lost_gate = rec_true & (pred == y) & (routed == GEN)
    u_routed, _, _ = route(P, 0.0)
    u_cont = (np.isin(u_routed, RIDX) & (u_routed != y)).sum()
    row = {'tau': tau, 'items': len(y), 'admitted': int(adm.sum()),
           'admitted_%': 100 * adm.mean(),
           'contaminants': int(cont.sum()),
           'contamination_%': 100 * cont.sum() / max(adm.sum(), 1),
           'intercepted': int(u_cont - cont.sum()),
           'recyclables_lost': int(lost.sum()), 'lost_to_gate': int(lost_gate.sum())}
    for c in RECYCLABLE:
        ci = CLASSES.index(c); s = routed == ci
        row[f'{c}_admitted'] = int(s.sum())
        row[f'{c}_purity_%'] = 100 * (y[s] == ci).mean() if s.any() else np.nan
    for kk in ([k] if k else COST_RATIOS):
        row[f'cost_k{kk}'] = kk * int(cont.sum()) + int(lost.sum())
    return row

def threshold_table(P, y, thresholds=THRESHOLDS):
    return pd.DataFrame([gate_metrics(P, y, t) for t in thresholds])

def choose_tau(P, y, k, grid=np.round(np.arange(0.0, 0.991, 0.01), 2)):
    costs = [gate_metrics(P, y, t, k)[f'cost_k{k}'] for t in grid]
    return float(grid[int(np.argmin(costs))])   # ties -> lowest tau (least diversion)

def classifier_metrics(P, y):
    pred = P.argmax(1)
    p, r, f, s = precision_recall_fscore_support(y, pred, labels=range(len(CLASSES)), zero_division=0)
    df = pd.DataFrame({'precision': p, 'recall': r, 'f1': f, 'support': s}, index=CLASSES)
    return df, accuracy_score(y, pred), confusion_matrix(y, pred, labels=range(len(CLASSES)))

def evaluate_seed(seed, logp_val, y_val, logp_test, y_test, latency=None):
    T = fit_temperature(logp_val, y_val)
    out = {'seed': seed, 'T': T, 'y_test': y_test, 'latency_s': latency}
    out['P_test'] = {'raw': softmax_T(logp_test, 1.0), 'cal': softmax_T(logp_test, T)}
    P_val = {'raw': softmax_T(logp_val, 1.0), 'cal': softmax_T(logp_val, T)}
    out['cls'], out['acc'], out['cm'] = classifier_metrics(out['P_test']['raw'], y_test)
    out['calib'] = {m: {'nll': nll(logp_test, y_test, 1.0 if m == 'raw' else T),
                        'ece': ece(out['P_test'][m], y_test)[0]} for m in ('raw', 'cal')}
    out['tables'] = {m: threshold_table(out['P_test'][m], y_test) for m in ('raw', 'cal')}
    sel = []
    for m in ('raw', 'cal'):
        for k in COST_RATIOS:
            tau = choose_tau(P_val[m], y_val, k)
            g = gate_metrics(out['P_test'][m], y_test, tau, k)
            u = gate_metrics(out['P_test'][m], y_test, 0.0, k)
            sel.append({'seed': seed, 'confidence': m, 'k': k, 'tau_theory': k / (k + 1), 'tau_val': tau,
                        'ungated_contamination_%': u['contamination_%'], 'gated_contamination_%': g['contamination_%'],
                        'ungated_admitted_%': u['admitted_%'], 'gated_admitted_%': g['admitted_%'],
                        'intercepted': g['intercepted'], 'lost_to_gate': g['lost_to_gate'],
                        'ungated_cost': u[f'cost_k{k}'], 'gated_cost': g[f'cost_k{k}']})
    out['selected'] = pd.DataFrame(sel)
    return out

## 6. Run every seed

In [ ]:
results = []
for seed in SEEDS:
    print(f'===== seed {seed} =====')
    root, parts = make_split(seed)
    model = train_seed(seed, root)
    lv, yv, _ = predict_logp(model, parts['val'])
    lt, yt, lat = predict_logp(model, parts['test'])
    results.append(evaluate_seed(seed, lv, yv, lt, yt, lat))
    print(f"accuracy {results[-1]['acc']:.3f} | T = {results[-1]['T']:.2f}")

## 7. Tables
Mean ± standard deviation across seeds. Per-seed threshold tables are also saved.

In [ ]:
def pm(series_list, fmt='{:.1f}'):
    a = np.array(series_list, dtype=float)
    return [fmt.format(m) + ' ± ' + fmt.format(s) for m, s in zip(np.nanmean(a, 0), np.nanstd(a, 0))]

def save(df, name, index=True):
    df.to_csv(OUT / f'{name}.csv', index=index)
    (OUT / f'{name}.md').write_text(df.to_markdown(index=index))
    display(df)

def make_tables(results):
    # classifier
    cls = pd.DataFrame(index=CLASSES + ['accuracy'])
    for col in ['precision', 'recall', 'f1']:
        cls[col] = pm([r['cls'][col].values * 100 for r in results]) + ['']
    cls.loc['accuracy'] = [pm([[r['acc'] * 100] for r in results])[0], '', '']
    cls['support (seed 0)'] = list(results[0]['cls']['support']) + [int(results[0]['cls']['support'].sum())]
    save(cls, 'table_classifier_metrics')
    cm = pd.DataFrame(results[0]['cm'], index=['true ' + c for c in CLASSES], columns=['pred ' + c for c in CLASSES])
    save(cm, 'table_confusion_matrix_seed0')
    # calibration
    cal = pd.DataFrame({
        'temperature T': pm([[r['T']] for r in results], '{:.2f}') * 2,
        'NLL': [pm([[r['calib'][m]['nll']] for r in results], '{:.3f}')[0] for m in ('raw', 'cal')],
        'ECE (%)': [pm([[100 * r['calib'][m]['ece']] for r in results])[0] for m in ('raw', 'cal')]},
        index=['raw softmax', 'temperature-scaled'])
    save(cal, 'table_calibration')
    # threshold tables
    cols = ['tau', 'admitted', 'admitted_%', 'paper_purity_%', 'plastic_purity_%', 'contamination_%',
            'contaminants', 'intercepted', 'recyclables_lost', 'lost_to_gate'] + [f'cost_k{k}' for k in COST_RATIOS]
    for m in ('raw', 'cal'):
        for r in results:
            r['tables'][m][cols].round(2).to_csv(OUT / f'table_threshold_{m}_seed{r["seed"]}.csv', index=False)
        mean = pd.DataFrame({c: pm([r['tables'][m][c].values for r in results], '{:.2f}' if c == 'tau' else '{:.1f}')
                             for c in cols[1:]})
        mean.insert(0, 'tau', ['ungated' if t == 0 else t for t in THRESHOLDS])
        save(mean, f'table_threshold_{m}_mean', index=False)
    # selected operating points
    sel = pd.concat([r['selected'] for r in results])
    agg = sel.groupby(['confidence', 'k']).agg(['mean', 'std']).drop(columns='seed')
    agg.columns = [f'{a}_{b}' for a, b in agg.columns]
    save(agg.round(2), 'table_selected_tau_test')
    return sel

sel_all = make_tables(results)
if results[0]['latency_s']:
    print(f"Colab GPU inference time: {1000*np.mean([r['latency_s'] for r in results]):.1f} ms/image "
          "(NOT the Raspberry Pi - measure on the Pi separately)")

## 8. Figures (from seed 0 unless stated)

In [ ]:
def make_figures(results):
    r0 = results[0]; y = r0['y_test']
    # Fig A: confusion matrix
    fig, ax = plt.subplots(figsize=(3.6, 3.2))
    ax.imshow(r0['cm'], cmap='Blues')
    for i in range(3):
        for j in range(3):
            ax.text(j, i, r0['cm'][i, j], ha='center', va='center',
                    color='white' if r0['cm'][i, j] > r0['cm'].max() / 2 else 'black')
    ax.set_xticks(range(3), CLASSES); ax.set_yticks(range(3), CLASSES)
    ax.set_xlabel('Predicted class'); ax.set_ylabel('True class'); ax.set_title('Confusion matrix (test, seed 0)')
    fig.tight_layout(); fig.savefig(OUT / 'fig_confusion_matrix.png'); plt.show()

    # Fig B: confidence of correct vs incorrect predictions (the decisive figure)
    fig, axes = plt.subplots(1, 2, figsize=(7, 2.8), sharey=True)
    for ax, m, title in zip(axes, ('raw', 'cal'), ('Raw softmax', f"Temperature-scaled (T={r0['T']:.2f})")):
        P = r0['P_test'][m]; ok = P.argmax(1) == y; c = P.max(1); b = np.linspace(1 / 3, 1, 21)
        ax.hist(c[ok], bins=b, alpha=.65, label=f'correct (n={ok.sum()})', color='#3B6EA5')
        ax.hist(c[~ok], bins=b, alpha=.75, label=f'incorrect (n={(~ok).sum()})', color='#C8553D')
        ax.set_yscale('log'); ax.set_xlabel('Confidence p'); ax.set_title(title); ax.legend(fontsize=8)
    axes[0].set_ylabel('Items (log scale)')
    fig.tight_layout(); fig.savefig(OUT / 'fig_confidence_separation.png'); plt.show()

    # Fig C: reliability diagram
    fig, ax = plt.subplots(figsize=(3.6, 3.4))
    ax.plot([0, 1], [0, 1], '--', color='grey', lw=1, label='perfect calibration')
    for m, col, lab in (('raw', '#C8553D', 'raw'), ('cal', '#3B6EA5', 'temperature-scaled')):
        e, rows = ece(r0['P_test'][m], y)
        if rows:
            cx, ac, _ = zip(*rows); ax.plot(cx, ac, 'o-', color=col, ms=4, label=f'{lab} (ECE {100*e:.1f}%)')
    ax.set_xlabel('Mean confidence'); ax.set_ylabel('Accuracy'); ax.set_xlim(.3, 1); ax.set_ylim(0, 1.02)
    ax.legend(fontsize=8); ax.set_title('Reliability (test, seed 0)')
    fig.tight_layout(); fig.savefig(OUT / 'fig_reliability.png'); plt.show()

    # Fig D: contamination vs admitted fraction (risk-coverage), all seeds
    fig, ax = plt.subplots(figsize=(4.2, 3.2))
    grid = np.round(np.arange(0, 0.996, 0.005), 3)
    for m, col, lab in (('raw', '#C8553D', 'raw softmax'), ('cal', '#3B6EA5', 'temperature-scaled')):
        for i, r in enumerate(results):
            rows = [gate_metrics(r['P_test'][m], r['y_test'], t) for t in grid]
            ax.plot([q['admitted_%'] for q in rows], [q['contamination_%'] for q in rows], color=col, alpha=.8,
                    lw=1.2, label=lab if i == 0 else None)
        u = gate_metrics(r0['P_test'][m], y, 0.0)
    ax.scatter([u['admitted_%']], [u['contamination_%']], color='black', zorder=5, label='ungated (seed 0)')
    ax.set_xlabel('Items admitted to recycling (%)'); ax.set_ylabel('Recycling contamination (%)')
    ax.set_title('Contamination vs. coverage'); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(OUT / 'fig_contamination_vs_coverage.png'); plt.show()

    # Fig E: expected cost vs tau for each cost ratio (seed 0, calibrated)
    fig, ax = plt.subplots(figsize=(4.2, 3.2))
    grid = np.round(np.arange(0, 0.991, 0.01), 2)
    for k, col in zip(COST_RATIOS, ('#7A9E7E', '#E0A458', '#3B6EA5')):
        c = np.array([gate_metrics(r0['P_test']['cal'], y, t, k)[f'cost_k{k}'] for t in grid], float)
        ax.plot(grid, c / c[0], color=col, label=f'k = {k}')
        ax.axvline(k / (k + 1), color=col, ls=':', lw=1)
    ax.set_xlabel('Gate threshold τ'); ax.set_ylabel('Expected cost relative to ungated')
    ax.set_title('Cost vs. threshold (dotted: τ* = k/(k+1))'); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(OUT / 'fig_cost_vs_threshold.png'); plt.show()

make_figures(results)

## 9. (Optional) Evaluate on your own line images
If your team still has photos captured on the real sorting line, upload them as a zip containing `paper/`, `plastic/`, `general/` folders and set `OWN_ZIP`. They are evaluated with the seed-0 model, its temperature, and the τ chosen on validation for `K_MAIN`. This is the closest you can get to the real line without the hardware.

In [ ]:
OWN_ZIP = None   # e.g. 'line_images.zip' after uploading it in the Files panel
if OWN_ZIP:
    shutil.rmtree('own', ignore_errors=True); zipfile.ZipFile(OWN_ZIP).extractall('own')
    own = pd.DataFrame([(str(p), p.parent.name) for p in Path('own').rglob('*')
                        if p.suffix.lower() in ('.jpg', '.jpeg', '.png') and p.parent.name in CLASSES],
                       columns=['path', 'cls'])
    print(own.cls.value_counts())
    m0 = YOLO('runs/seed0/weights/best.pt'); lo, yo, _ = predict_logp(m0, own)
    r0 = results[0]; tau = float(sel_all.query('seed == @SEEDS[0] and confidence == "cal" and k == @K_MAIN').tau_val.iloc[0])
    Po = softmax_T(lo, r0['T'])
    save(classifier_metrics(Po, yo)[0].round(3), 'table_own_images_classifier')
    save(pd.DataFrame([gate_metrics(Po, yo, 0.0), gate_metrics(Po, yo, tau)]).round(2), 'table_own_images_gate', index=False)

## 10. Export: results zip + model for the Raspberry Pi

In [ ]:
best = 'runs/seed0/weights/best.pt'
shutil.copy(best, OUT / 'econexus_yolov8n_cls_best.pt')
tau_main = float(sel_all.query('seed == @SEEDS[0] and confidence == "cal" and k == @K_MAIN').tau_val.iloc[0])
cfg = {'model': 'econexus_yolov8n_cls_best.pt', 'temperature': results[0]['T'],
       'threshold_tau': tau_main, 'cost_ratio_k': K_MAIN, 'classes': CLASSES,
       'note': 'T and tau were chosen on the validation split of seed 0'}
(OUT / 'gate_config.json').write_text(json.dumps(cfg, indent=2)); print(cfg)
shutil.make_archive('econexus_results', 'zip', OUT)
from google.colab import files; files.download('econexus_results.zip')

## Writing it up honestly
- Call this an **offline evaluation on TrashNet**, not a test of the physical line.
- Report the dataset split table, classifier table, confusion matrix, calibration table, the threshold table, and the selected-τ table (τ chosen on validation, reported on test).
- The **confidence-separation figure** is the decisive one (paper, Section V-D). If correct and incorrect predictions overlap heavily, say the gate helps little.
- State the cost ratio k you used and why; show results for all k values rather than only the best one.
- Report Raspberry Pi inference time only if you measure it on a Pi; the Colab time is not comparable.